In [ ]:
import xllim
import kernelo
import numpy as np
import time
import json
import logging
logging.getLogger().setLevel(logging.INFO)

## Physical model

In [ ]:
# Get geometries from JSON file
with open("../dataRef/JSC1_BRDF.json", "r") as f:
    data = json.load(f)
geometries = np.array(data["JSC1_analogue"]["geometries"], dtype=float)
print(geometries.shape)

# Define other Hapke arguments
variant = "2002"
adapter = "four"
theta_bar_scaling = 30.0
b0 = 0
h = 0.1

# Create Hapke model
physical_model_xllim = xllim.HapkeModel(geometries, variant, adapter, theta_bar_scaling, b0, h)

adapterConfig = kernelo.FourParamsHapkeAdapterConfig(b0, h)
physical_model_kernelo = kernelo.HapkeModelConfig(variant, adapterConfig, geometries, theta_bar_scaling).create()

In [ ]:
L = physical_model_xllim.getDimensionX()
D = physical_model_xllim.getDimensionY()
print("Test model transform a vector X of dimension L = {} into a vector Y of dimension D = {}\n".format(L, D))

x = np.ones(L)
x_physic = physical_model_xllim.toPhysic(x)
x_mat = physical_model_xllim.fromPhysic(x_physic)
print("physical_model_xllim.toPhysic({}) = {}".format(x, x_physic))
print("physical_model_xllim.fromPhysic({}) = {}\n".format(x_physic, x_mat))

x_physic = physical_model_kernelo.to_physic(x)
x_mat = physical_model_kernelo.from_physic(x_physic)
print("physical_model_kernelo.to_physic({}) = {}".format(x, x_physic))
print("physical_model_kernelo.from_physic({}) = {}\n".format(x_physic, x_mat))

x = np.ones(L)
y = physical_model_xllim.F(x)
print("physical_model_xllim.F({}) = ".format(x))
print(y)

x = np.ones(L)
y = physical_model_kernelo.F(x)
print("physical_model_kernelo.F({}) = ".format(x))
print(y)


In [ ]:
N_test = 100000

start = time.time()
for i in range(N_test):
    x = np.random.rand(L)
    y = physical_model_xllim.F(x)
time_xllim = time.time() - start

start = time.time()
for i in range(N_test):
    x = np.random.rand(L)
    y = physical_model_kernelo.F(x)
time_kernelo = time.time() - start

print('time xllim = {} sec'.format(time_xllim))
print('time kernelo = {} sec'.format(time_kernelo))

## Generate Dataset

In [ ]:
N = 10_000                                            # number of generated observation
generator_type = "sobol"                            # the type of the generator used to generate x_gen matrix values
covariance = np.ones(D) * 1e-5                      # vector of dimension D coresponding to the y_i variances.
seed = 12345                                        # seed number for random generators

In [ ]:
start = time.time()
x_gen, y_gen = physical_model_xllim.genData(N, generator_type, covariance, seed)
time_xllim = time.time() - start
print(time_xllim)


In [ ]:
start = time.time()
stat_model_kernelo = kernelo.GaussianStatModelConfig(generator_type, physical_model_kernelo, covariance, seed).create()
x_gen_ker, y_gen_ker = stat_model_kernelo.gen_data(N)
time_kernelo = time.time() - start

In [ ]:
print(x_gen.shape)
print(y_gen.shape)
print(x_gen_ker.shape)
print(y_gen_ker.shape)
print(np.all(x_gen_ker == x_gen))
print(np.allclose(y_gen_ker, y_gen))

print('time xllim = {} sec'.format(time_xllim))
print('time kernelo = {} sec'.format(time_kernelo))

## GLLiM

In [ ]:
K = 50
gamma_type = 'Full'
sigma_type = 'Diag'
seed = 12345678

# initialisation parameters
gllim_em_iteration = 10
gllim_em_floor = 1e-12
gmm_kmeans_iteration = 1
gmm_em_iteration = 1
gmm_floor = 1e-12
nb_experiences = 10

# training parameters
train_max_iteration = 200
train_ratio_ll = -1000 #1e-5
train_floor = 1e-12


gllim_xllim = xllim.GLLiM(K, D, L, gamma_type.lower(), sigma_type.lower(), 0) # hidden_values = 0

learningConfig = kernelo.EMLearningConfig(train_max_iteration, train_ratio_ll, train_floor)
# learningConfig=kernelo.GMMLearningConfig(train_max_iteration, train_ratio_ll, train_floor)
# initConfig = kernelo.MultInitConfig(seed=123456789, nb_iter_EM=10, nb_experiences=10, gmmLearningConfig=oldker.GMMLearningConfig(15, 10, 1e-12))
initConfig = kernelo.MultInitConfig(seed=seed, nb_iter_EM=gllim_em_iteration, nb_experiences=nb_experiences, gmmLearningConfig=kernelo.GMMLearningConfig(gmm_kmeans_iteration, gmm_em_iteration, gmm_floor))
gllim_kernelo= kernelo.GLLiM(D, L, K, gamma_type, sigma_type, initConfig, learningConfig)

## Initialisation

In [ ]:
x_gen_T = np.array(x_gen.T)
y_gen_T = np.array(y_gen.T)
start = time.time()
gllim_xllim.initialize( x_gen_T, y_gen_T, gllim_em_iteration, gllim_em_floor, gmm_kmeans_iteration, gmm_em_iteration, gmm_floor, nb_experiences, seed, 1)
time_xllim = time.time() - start
gllim_params_0_xllim = gllim_xllim.getParams()

In [ ]:
start = time.time()
gllim_kernelo.initialize(x_gen, y_gen)
time_kernelo= time.time() - start
gllim_params_0_kernelo = gllim_kernelo.exportModel()

In [ ]:
print('time xllim = {} sec'.format(time_xllim))
print('time kernelo = {} sec'.format(time_kernelo))

# check if weights are the same (may be not in the same order)
print("Pi xllim/kernelo:")
print(gllim_params_0_xllim.Pi)
print(gllim_params_0_kernelo.Pi)

k = 0 # some component with same weight

print("")
print("same A ?", np.all(np.isclose(gllim_params_0_xllim.A[k], gllim_params_0_kernelo.A[k])))
print("same B ?", np.all(np.isclose(gllim_params_0_xllim.B[k], gllim_params_0_kernelo.B[:,k])))
print("same C ?", np.all(np.isclose(gllim_params_0_xllim.C[k], gllim_params_0_kernelo.C[:,k])))
print("same Gamma ?", np.all(np.isclose(gllim_params_0_xllim.Gamma[k], gllim_params_0_kernelo.Gamma[k])))
print("same Sigma ?", np.all(np.isclose(gllim_params_0_xllim.Sigma[k], np.diag(gllim_params_0_kernelo.Sigma[k]))))
print("")

# print(gllim_params_0_xllim.A[k])
# print(gllim_params_0_kernelo.A[k])
# print(gllim_params_0_xllim.B[k])
# print(gllim_params_0_kernelo.B[:,k])
print(gllim_params_0_xllim.C)
print(gllim_params_0_kernelo.C)
# print(gllim_params_0_xllim.Gamma[k])
# print(gllim_params_0_kernelo.Gamma[k]) # full
# print(gllim_params_0_xllim.Sigma[k])
# print(np.diag(gllim_params_0_kernelo.Sigma[k])) # diag


## Training

In [ ]:
start = time.time()
gllim_xllim.train(x_gen_T, y_gen_T, train_max_iteration, train_ratio_ll, train_floor, 1)
gllim_params_xllim = gllim_xllim.getParams()
time_xllim = time.time() - start

In [ ]:
start = time.time()
gllim_kernelo.train(x_gen, y_gen)
time_kernelo = time.time() - start
gllim_params_kernelo = gllim_kernelo.exportModel()

In [ ]:
print('time xllim = {} sec'.format(time_xllim))
print('time kernelo = {} sec'.format(time_kernelo))

# check if weights are the same (may be not in the same order)
print("Pi xllim/kernelo:")
print(gllim_params_xllim.Pi)
print(gllim_params_kernelo.Pi)

k = 0 # some component with same weight

print("")
print("same A ?", np.all(np.isclose(gllim_params_xllim.A[k], gllim_params_kernelo.A[k])))
print("same B ?", np.all(np.isclose(gllim_params_xllim.B[k], gllim_params_kernelo.B[:,k])))
print("same C ?", np.all(np.isclose(gllim_params_xllim.C[k], gllim_params_kernelo.C[:,k])))
print("same Gamma ?", np.all(np.isclose(gllim_params_xllim.Gamma[k], gllim_params_kernelo.Gamma[k])))
print("same Sigma ?", np.all(np.isclose(gllim_params_xllim.Sigma[k], np.diag(gllim_params_kernelo.Sigma[k]))))
print("")

# print(gllim_params_xllim.A[k])
# print(gllim_params_kernelo.A[k])
# print(gllim_params_xllim.B[k])
# print(gllim_params_kernelo.B[:,k])
print(gllim_params_xllim.C[k])
print(gllim_params_kernelo.C[:,k])
# print(gllim_params_xllim.Gamma[k])
# print(gllim_params_kernelo.Gamma[k]) # full
# print(gllim_params_xllim.Sigma[k])
# print(np.diag(gllim_params_kernelo.Sigma[k])) # diag

## Inversion

In [ ]:
theta_star_xllim = gllim_xllim.getInverse()
theta_star_kernelo = gllim_kernelo.getInverse()

In [ ]:
# check if weights are the same (may be not in the same order)
print("Pi xllim/kernelo:")
print(theta_star_xllim.Pi)
print(theta_star_kernelo.Pi)

k = 0 # some component with same weight

print("")
print("same A ?", np.all(np.isclose(theta_star_xllim.A[k], theta_star_kernelo.A[k])))
print("same B ?", np.all(np.isclose(theta_star_xllim.B[k], theta_star_kernelo.B[:,k])))
print("same C ?", np.all(np.isclose(theta_star_xllim.C[k], theta_star_kernelo.C[:,k])))
print("same Gamma ?", np.all(np.isclose(theta_star_xllim.Gamma[k], theta_star_kernelo.Gamma[k])))
print("same Sigma ?", np.all(np.isclose(theta_star_xllim.Sigma[k], theta_star_kernelo.Sigma[k])))
print("")

# print(theta_star_xllim.A[k])
# print(theta_star_kernelo.A[k])
print(theta_star_xllim.B[k])
print(theta_star_kernelo.B[:,k])
# print(theta_star_xllim.C[k])
# print(theta_star_kernelo.C[:,k])
# print(theta_star_xllim.Gamma[k])
# print(theta_star_kernelo.Gamma[k]) # full
# print(theta_star_xllim.Sigma[k])
# print(theta_star_kernelo.Sigma[k]) # diag

## Prediction

In [ ]:
predicator_kernelo = kernelo.PredictionConfig(2, 2, 1e-10, gllim_kernelo).create() # create Kernelo predictor once

#### Prediction on training dataset

In [ ]:
err_kernelo = 0
err_xllim = 0
start = time.time()
prediction_dataset_xllim = gllim_xllim.inverseDensities(np.array(y_gen.T), np.zeros(D), 2, 1e-10) # vectorized
time_xllim = time.time() - start

start = time.time()
prediction_dataset_kernelo = []
for i in range(N):
    pred = predicator_kernelo.predict(y_gen[i], np.zeros(D))
    prediction_dataset_kernelo.append(pred)
time_kernelo = time.time() - start

for i in range(N):
    err_xllim += np.linalg.norm(prediction_dataset_xllim.fullGMM.mean[i] - x_gen[i]) / np.linalg.norm(x_gen[i])
    err_kernelo += np.linalg.norm(prediction_dataset_kernelo[i].meansPred.mean - x_gen[i]) / np.linalg.norm(x_gen[i])

print('time xllim = {} sec'.format(time_xllim))
print('time kernelo = {} sec'.format(time_kernelo))

print("err_xllim = {}".format(err_xllim))
print("err_kernelo = {}".format(err_kernelo))

In [ ]:
for i in range(5,9):
    print(x_gen[i])
    print(prediction_dataset_kernelo[i].meansPred.mean)
    print(prediction_dataset_xllim.fullGMM.mean[i])
    print("")

#### Prediction on JSC1 data

In [ ]:
# prediction on real JSC1 data

import matplotlib.pyplot as plt

# Get geometries from JSON file
with open("../dataRef/JSC1_BRDF.json", "r") as f:
    data = json.load(f)
wavelengths = np.array(data["JSC1_analogue"]["wavelengths"])
reflectances = np.array(data["JSC1_analogue"]["reflectance_JSC1"])
N_obs = wavelengths.shape[0]

print("There is {} observations !".format(N_obs))
print("wavelengths has shape : {}".format(wavelengths.shape))
print("reflectances has shape : {}".format(reflectances.shape))

observations = np.array(reflectances[:,0,:].T)
incertitudes = np.array(reflectances[:,1,:].T)

print("observations has shape : {}".format(observations.shape))
print("incertitudes has shape : {}".format(incertitudes.shape))
print(observations)
print(incertitudes)

In [ ]:
err_y_xllim = []
err_y_kernelo = []

start = time.time()
predictions_obs_xllim = gllim_xllim.inverseDensities(observations, incertitudes, 2, 1e-10, 0)
time_xllim = time.time() - start

start = time.time()
predictions_obs_kernelo = []
prediction_dataset_kernelo = []
for i in range(N_obs):
    pred = predicator_kernelo.predict(observations[:,i], incertitudes[:,i])
    predictions_obs_kernelo.append(pred)
time_kernelo = time.time() - start

for i in range(N_obs):
    err_y_kernelo.append(np.linalg.norm(physical_model_kernelo.F(predictions_obs_kernelo[i].meansPred.mean) - observations[:,i]) / np.linalg.norm(observations[:,i]))
    err_y_xllim.append(np.linalg.norm(physical_model_xllim.F(predictions_obs_xllim.fullGMM.mean[i]) - observations[:,i]) / np.linalg.norm(observations[:,i]))

print('time xllim = {} sec'.format(time_xllim))
print('time kernelo = {} sec'.format(time_kernelo))

print("err_y_xllim = {}".format(sum(err_y_xllim)/N_obs))
print("err_y_kernelo = {}".format(sum(err_y_kernelo)/N_obs))

In [ ]:
idx_y = np.random.choice(N_obs)

print("{}".format(observations[:,idx_y]))
print("{} = F({})".format(physical_model_kernelo.F(predictions_obs_kernelo[idx_y].meansPred.mean), predictions_obs_kernelo[idx_y].meansPred.mean))
print("{} = F({})".format(physical_model_xllim.F(predictions_obs_xllim.fullGMM.mean[idx_y]), predictions_obs_xllim.fullGMM.mean[idx_y]))
print("")


In [ ]:
### Plot some Y parameters with respects to wavelengths ###
fig, axs = plt.subplots(1, 3, figsize=(30, 5))
fig.suptitle("Reflectances", fontsize=16)
y_idx_list = [10, 20, 60]
                       
for d in range(len(y_idx_list)):
    axs[d].plot(np.arange(D), observations[:,y_idx_list[d]], 'k-', label='obs')

    pred = predicator_kernelo.predict(observations[:,y_idx_list[d]], incertitudes[:,y_idx_list[d]])
    y_kernelo = physical_model_kernelo.F(pred.meansPred.mean)
    axs[d].plot(np.arange(D), y_kernelo, 'r-', label='kernelo')

    y_xllim = physical_model_xllim.F(predictions_obs_xllim.fullGMM.mean[y_idx_list[d]])
    axs[d].plot(np.arange(D), y_xllim, 'g-', label='xllim')

    axs[d].legend()
    axs[d].set_xlabel("Observations")
    axs[d].set_title('Y_'+ str(y_idx_list[d]))
    
plt.show()

In [ ]:
plt.figure()             
plt.plot(wavelengths, err_y_kernelo, 'r-', label='kernelo')
plt.plot(wavelengths, err_y_xllim, 'g-', label='xllim')
plt.legend()
plt.xlabel("Wavelengths")
plt.title('Relative error')
    
plt.show()

In [ ]:
### Plot some Y parameters with respects to wavelengths ###
fig, axs = plt.subplots(1, 4, figsize=(30, 5))
fig.suptitle("Parameters", fontsize=16)
print(predictions_obs_xllim.mergedGMM.means.shape)

for l in range(L):
    axs[l].plot(wavelengths, [x.meansPred.mean[l] for x in predictions_obs_kernelo], 'r-', label='kernelo')
    axs[l].plot(wavelengths, predictions_obs_xllim.fullGMM.mean[:,l], 'g-', label='xllim')
    axs[l].legend()
    axs[l].set_xlabel("Wavelengths")
    axs[l].set_title('X_'+ str(l))
    
plt.show()

# Sampling

In [ ]:
# Covariance on the target density
covariance = np.ones(D) * 1e-5

# Importance sampling parameters
N_0 = 1000
B = 50
J = 20

In [ ]:
err_y_is_kernelo = []
err_y_is_xllim = []

is_results_kernelo = []
start = time.time()
imis_sampler = kernelo.ImisConfig(N_0, B, J, stat_model_kernelo).create()
for i in range(N_obs):
    proposition_gmm_kernelo = kernelo.GaussianMixturePropositionConfig(
        predictions_obs_kernelo[i].meansPred.gmm_weights,
        predictions_obs_kernelo[i].meansPred.gmm_means,
        predictions_obs_kernelo[i].meansPred.gmm_covs
        ).create()
    imis_sampler = kernelo.ImisConfig(N_0, B, J, stat_model_kernelo).create()
    is_results_kernelo.append(imis_sampler.execute(proposition_gmm_kernelo, observations[:,i], incertitudes[:,i]))
time_kernelo = time.time() - start

y_obs_noised_T = np.array(observations.T)
y_obs_noise_T = np.array(incertitudes.T)

start = time.time()
is_results_xllim = physical_model_xllim.importanceSampling(predictions_obs_xllim.mergedGMM, y_obs_noised_T, y_obs_noise_T, N_0, B, J, covariance, verbose=0)
time_xllim = time.time() - start

for i in range(N_obs):
    err_y_is_kernelo.append(np.linalg.norm(physical_model_kernelo.F(is_results_kernelo[i].mean) - observations[:,i]) / np.linalg.norm(observations[:,i]))
    err_y_is_xllim.append(np.linalg.norm(physical_model_xllim.F(is_results_xllim.predictions[:,i]) - observations[:,i]) / np.linalg.norm(observations[:,i]))


print('time xllim = {} sec'.format(time_xllim))
print('time kernelo = {} sec'.format(time_kernelo))

print("err_y_is_kernelo = {}".format(sum(err_y_is_kernelo)/N_obs))
print("err_y_is_xllim = {}".format(sum(err_y_is_xllim)/N_obs))

In [ ]:
### Plot some Y parameters with respects to wavelengths ###
fig, axs = plt.subplots(1, 3, figsize=(30, 5))
fig.suptitle("Reflectances (IMIS)", fontsize=16)
y_idx_list = [15, 21, 60]
                       
for d in range(len(y_idx_list)):
    n = y_idx_list[d]
    axs[d].plot(np.arange(D), observations[:,n], 'k-', label='obs')

    y_kernelo = physical_model_kernelo.F(is_results_kernelo[n].mean)
    axs[d].plot(np.arange(D), y_kernelo, 'r-', label='kernelo')

    y_xllim = physical_model_xllim.F(is_results_xllim.predictions[:,n])
    axs[d].plot(np.arange(D), y_xllim, 'g-', label='xllim')

    axs[d].legend()
    axs[d].set_xlabel("Observations")
    axs[d].set_title('Y_'+ str(n))
    
plt.show()

In [ ]:
plt.figure()             
plt.plot(wavelengths, err_y_is_kernelo, 'r-', label='kernelo')
plt.plot(wavelengths, err_y_is_xllim, 'g-', label='xllim')
plt.legend()
plt.xlabel("Wavelengths")
plt.title('Relative error  (IMIS)')
    
plt.show()

In [ ]:
### Plot some Y parameters with respects to wavelengths ###
fig, axs = plt.subplots(1, 4, figsize=(30, 5))
fig.suptitle("Parameters (IMIS)", fontsize=16)
print(predictions_obs_xllim.mergedGMM.means.shape)

for l in range(L):
    axs[l].plot(wavelengths, [x.mean[l] for x in is_results_kernelo], 'r-', label='kernelo')
    axs[l].plot(wavelengths, is_results_xllim.predictions[l], 'g-', label='xllim')
    axs[l].legend()
    axs[l].set_xlabel("Wavelengths")
    axs[l].set_title('X_'+ str(l))
    
plt.show()